# Audio Teacher Baseline: Wav2Vec 2.0 Base & HuBERT Base

This notebook establishes baselines for two audio teacher models we plan to distil:
- **Wav2Vec 2.0 Base** (`facebook/wav2vec2-base`) — self-supervised CNN + Transformer, trained on LibriSpeech 960h
- **HuBERT Base** (`facebook/hubert-base-ls960`) — offline k-means clustering targets + masked prediction, also LibriSpeech 960h

We measure: parameter count, memory footprint, hidden representation shape, and inference latency.  
These numbers define the distillation target we need to beat on the edge.

## 0. Environment Setup

**Recommended**: create a dedicated venv for this project (don't reuse `img_venv`):

```powershell
# from repo root
python -m venv .venv
.venv\Scripts\Activate.ps1
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
pip install transformers datasets soundfile librosa evaluate jiwer
pip install ipykernel notebook
python -m ipykernel install --user --name mmkd --display-name "mmkd (Python 3.12)"
```

Then select the `mmkd` kernel in VS Code / Jupyter before running this notebook.

In [ ]:
# Quick check — run once to verify the environment
import importlib, sys

required = ["torch", "torchaudio", "transformers", "datasets", "soundfile", "librosa", "evaluate"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print(f"Missing packages: {missing}")
    print("Run the pip install block above, then restart the kernel.")
else:
    import torch, torchaudio, transformers
    print(f"Python      : {sys.version}")
    print(f"PyTorch     : {torch.__version__}")
    print(f"torchaudio  : {torchaudio.__version__}")
    print(f"transformers: {transformers.__version__}")
    print(f"CUDA        : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

## 1. Shared Utilities

In [ ]:
import time, gc
import torch
import torchaudio
import numpy as np
import pandas as pd

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TARGET_SR = 16_000  # both models expect 16 kHz mono
WARMUP_RUNS = 5
BENCH_RUNS  = 50


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total_M": total / 1e6, "trainable_M": trainable / 1e6}


def model_size_mb(model):
    """Rough parameter storage size in FP32."""
    return sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6


def benchmark_latency(model, input_values, runs=BENCH_RUNS, warmup=WARMUP_RUNS):
    model.eval()
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(input_values)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(runs):
            _ = model(input_values)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        elapsed = (time.perf_counter() - t0) / runs * 1000  # ms
    return elapsed


print(f"Device: {DEVICE}")

## 2. Load Sample Audio

We use a short LibriSpeech clip from HuggingFace Datasets (streaming, ~few MB).  
No local audio file required.

In [ ]:
from datasets import load_dataset

# 'clean' split, streaming — only loads one example
ds = load_dataset("librispeech_asr", "clean", split="validation", streaming=True, trust_remote_code=True)
sample = next(iter(ds))

audio_array = np.array(sample["audio"]["array"], dtype=np.float32)
sr = sample["audio"]["sampling_rate"]
text_ref = sample["text"]

print(f"Reference text : {text_ref}")
print(f"Sample rate    : {sr} Hz")
print(f"Duration       : {len(audio_array)/sr:.2f}s  ({len(audio_array)} samples)")

In [ ]:
import librosa

# resample to 16 kHz if needed
if sr != TARGET_SR:
    audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=TARGET_SR)
    sr = TARGET_SR

# tensor: (1, T) — batch of 1
audio_tensor = torch.tensor(audio_array).unsqueeze(0).to(DEVICE)
print(f"Input tensor shape : {audio_tensor.shape}")
print(f"Input duration     : {audio_tensor.shape[-1] / TARGET_SR:.2f}s")

## 3. Wav2Vec 2.0 Base

Architecture at a glance:
- **Feature encoder**: 7-layer CNN (stride product = 320 → 20 ms frames at 16 kHz)
- **Context network**: 12-layer Transformer (d=768, 8 heads)
- **Quantiser**: product quantisation for contrastive self-supervised pretraining

For distillation purposes the important outputs are:
- `last_hidden_state` — final Transformer output, shape `(B, T', 768)`
- `hidden_states` — all 12 layer outputs (enable with `output_hidden_states=True`)

In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2Model

W2V_CKPT = "facebook/wav2vec2-base"

w2v_processor = Wav2Vec2Processor.from_pretrained(W2V_CKPT)
w2v_model = Wav2Vec2Model.from_pretrained(W2V_CKPT, output_hidden_states=True).to(DEVICE)
w2v_model.eval()

w2v_stats = count_params(w2v_model)
print(f"Parameters  : {w2v_stats['total_M']:.1f}M  (trainable {w2v_stats['trainable_M']:.1f}M)")
print(f"Storage (FP32): {model_size_mb(w2v_model):.1f} MB")

In [ ]:
# Preprocess: feature extraction + normalisation
inputs_w2v = w2v_processor(
    audio_array, sampling_rate=TARGET_SR, return_tensors="pt", padding=True
).to(DEVICE)

with torch.no_grad():
    out_w2v = w2v_model(**inputs_w2v)

print(f"last_hidden_state shape : {out_w2v.last_hidden_state.shape}")
print(f"  = (batch, frames, hidden_dim)")
print(f"Frame rate: {audio_tensor.shape[-1] / out_w2v.last_hidden_state.shape[1] / TARGET_SR * 1000:.1f} ms/frame")
print(f"Number of hidden layers returned: {len(out_w2v.hidden_states)}")

In [ ]:
# Per-layer output norms — useful for choosing which layer to distil from
print("Layer-wise L2 norm of hidden states (mean over batch & time):")
for i, hs in enumerate(out_w2v.hidden_states):
    norm = hs.norm(dim=-1).mean().item()
    print(f"  Layer {i:2d}: {norm:.3f}")

In [ ]:
w2v_latency = benchmark_latency(w2v_model, inputs_w2v)
print(f"Wav2Vec 2.0 Base inference latency: {w2v_latency:.2f} ms  (avg over {BENCH_RUNS} runs on {DEVICE})")

## 4. HuBERT Base

Architecture at a glance:
- **Feature encoder**: identical 7-layer CNN to Wav2Vec2 (same stride → same 20 ms frames)
- **Context network**: 12-layer Transformer (d=768, 8 heads) — same capacity as Wav2Vec2 Base
- **Pretraining objective**: offline k-means cluster IDs as pseudo-labels (BERT-style masked prediction)

Key difference from Wav2Vec2: no quantiser at inference; the entire model is a feature extractor.

In [ ]:
from transformers import HubertModel, Wav2Vec2FeatureExtractor

HUB_CKPT = "facebook/hubert-base-ls960"

hub_processor = Wav2Vec2FeatureExtractor.from_pretrained(HUB_CKPT)
hub_model = HubertModel.from_pretrained(HUB_CKPT, output_hidden_states=True).to(DEVICE)
hub_model.eval()

hub_stats = count_params(hub_model)
print(f"Parameters  : {hub_stats['total_M']:.1f}M  (trainable {hub_stats['trainable_M']:.1f}M)")
print(f"Storage (FP32): {model_size_mb(hub_model):.1f} MB")

In [ ]:
inputs_hub = hub_processor(
    audio_array, sampling_rate=TARGET_SR, return_tensors="pt", padding=True
).to(DEVICE)

with torch.no_grad():
    out_hub = hub_model(**inputs_hub)

print(f"last_hidden_state shape : {out_hub.last_hidden_state.shape}")
print(f"Frame rate: {audio_tensor.shape[-1] / out_hub.last_hidden_state.shape[1] / TARGET_SR * 1000:.1f} ms/frame")
print(f"Number of hidden layers returned: {len(out_hub.hidden_states)}")

In [ ]:
print("Layer-wise L2 norm of hidden states (mean over batch & time):")
for i, hs in enumerate(out_hub.hidden_states):
    norm = hs.norm(dim=-1).mean().item()
    print(f"  Layer {i:2d}: {norm:.3f}")

In [ ]:
hub_latency = benchmark_latency(hub_model, inputs_hub)
print(f"HuBERT Base inference latency: {hub_latency:.2f} ms  (avg over {BENCH_RUNS} runs on {DEVICE})")

## 5. Summary Comparison

In [ ]:
summary = pd.DataFrame([
    {
        "Model": "Wav2Vec 2.0 Base",
        "Checkpoint": W2V_CKPT,
        "Params (M)": f"{w2v_stats['total_M']:.1f}",
        "Size FP32 (MB)": f"{model_size_mb(w2v_model):.1f}",
        "Output dim": out_w2v.last_hidden_state.shape[-1],
        "Num layers": len(out_w2v.hidden_states) - 1,  # -1 for CNN embed layer
        "Frame rate (ms)": f"{audio_tensor.shape[-1] / out_w2v.last_hidden_state.shape[1] / TARGET_SR * 1000:.0f}",
        f"Latency on {DEVICE} (ms)": f"{w2v_latency:.1f}",
    },
    {
        "Model": "HuBERT Base",
        "Checkpoint": HUB_CKPT,
        "Params (M)": f"{hub_stats['total_M']:.1f}",
        "Size FP32 (MB)": f"{model_size_mb(hub_model):.1f}",
        "Output dim": out_hub.last_hidden_state.shape[-1],
        "Num layers": len(out_hub.hidden_states) - 1,
        "Frame rate (ms)": f"{audio_tensor.shape[-1] / out_hub.last_hidden_state.shape[1] / TARGET_SR * 1000:.0f}",
        f"Latency on {DEVICE} (ms)": f"{hub_latency:.1f}",
    },
])

summary.set_index("Model", inplace=True)
summary

## 6. Cosine Similarity Between Teacher Representations

Before designing the student we want to know how similar the two teachers' output spaces are.  
High similarity → easier to share a single student trunk; low similarity → may need separate heads.

In [ ]:
import torch.nn.functional as F

# Both outputs: (1, T', 768) — take mean over time → (1, 768)
w2v_cls = out_w2v.last_hidden_state.mean(dim=1)  # mean pooling
hub_cls = out_hub.last_hidden_state.mean(dim=1)

cos_sim = F.cosine_similarity(w2v_cls, hub_cls).item()
print(f"Cosine similarity between Wav2Vec2 and HuBERT mean-pooled representations: {cos_sim:.4f}")
print("(1.0 = identical direction, 0.0 = orthogonal, −1.0 = opposite)")

## 7. Notes for Distillation Design

| Observation | Implication |
|---|---|
| Both teachers ~94 M params, 360 MB FP32 | Student target: <10 M params / <40 MB for extreme edge |
| Frame rate 20 ms (stride 320 @ 16 kHz) | Student CNN must match this or use adapter projection |
| Hidden dim 768, 12 Transformer layers | Feature-based KD: map student hidden → teacher hidden via linear projector |
| HuBERT no quantiser at inference | Simpler teacher forward pass; wav2vec2 quantiser can be detached during distil |

**Next steps:**
1. Run ASR fine-tuned variants (`wav2vec2-base-960h`, `hubert-base-ls960`) for WER baseline
2. Decide which teacher (or both) to use for audio distillation
3. Begin student architecture design (MobileNet-style CNN + tiny Transformer)